# Harmonizome EvoKG Data Processing

**Input:** Intermediate CSVs generated by `part1_harmonizome`  
**Output:** Final merged KG CSVs in `Processed/` folder

### Output schema (all files):
`Head | Relation | Tail | Head_type | Tail_type | Source | KG_Source | Head_detail_name | Tail_detail_name | Head_ID_IS | Tail_ID_IS`

### Relation types produced:
- `Gene_Phenotype` → Gene–Phenotype
- `Gene_Gene` → Gene–Gene (PPI + TF targets + co-expression)
- `Gene_Tissue` → Gene–Tissue
- `Gene_Cellular Component` → Gene–GO Cellular Component
- `Gene_Disease` → Gene–Disease
- `Gene_ChemicalEntity` → Gene–Drug/Chemical
- `Gene_Protein` → Gene–Protein (UniProt)
- `Gene_Metabolite` → Gene–Metabolite (PubChem)
- `Gene_MolecularFunction` → Gene–GO Molecular Function
- `Gene_Pathway` → Gene–Pathway (Reactome)

## 0. Imports & Config

In [53]:
import os
import pandas as pd
import numpy as np
from glob import glob

# ── PATHS ─────────────────────────────────────────────────
# Input: intermediate CSVs generated by harmonizome_raw_to_csv.py
BASE_PATH = '/storage/Arushi/090526_EvoAge/kg_formation/data_collection/harmonizome/refine_harmonizome/new_refine_harmonizome'
RAW_DIR   = BASE_PATH

# Output: final merged KG CSVs
OUT_PATH  = '/storage/Arushi/090526_EvoAge/kg_formation/processed_data/harmonizome'
OUT_DIR   = OUT_PATH

# External reference databases
DB_DIR    = '/storage/Arushi/090526_EvoAge/kg_formation/data_collection/databases_for_mapping'

os.makedirs(OUT_DIR, exist_ok=True)
print(f'Input  (RAW_DIR) : {RAW_DIR}')
print(f'Output (OUT_DIR) : {OUT_DIR}')

# ── DESIRED OUTPUT COLUMNS ────────────────────────────────
DESIRED_COLS = [
    'Head', 'Relation', 'Tail', 'Head_type', 'Tail_type',
    'Source', 'KG_Source', 'Head_detail_name', 'Tail_detail_name',
    'Head_ID_IS', 'Tail_ID_IS'
]

def finalize(df, head_type, tail_type, head_id_is, tail_id_is):
    """Add metadata columns, filter NaNs, keep only valid columns, deduplicate."""
    df['Head_type']  = head_type
    df['Tail_type']  = tail_type
    df['Relation']   = head_type + '_' + tail_type
    df['KG_Source']  = 'Harmonizome'
    df['Head_ID_IS'] = head_id_is
    df['Tail_ID_IS'] = tail_id_is
    # Drop rows where Head or Tail is missing
    df = df[~df['Head'].isna() & ~df['Tail'].isna()]
    df = df[~df['Head_detail_name'].isna()]
    # Keep only desired columns that exist
    keep = [c for c in DESIRED_COLS if c in df.columns]
    df = df[keep].drop_duplicates()
    return df

def save(df, filename):
    path = os.path.join(OUT_DIR, filename)
    df.to_csv(path, index=False)
    print(f'  [SAVED] {filename}  ({len(df):,} rows)')

print('Config ready.')

Input  (RAW_DIR) : /storage/Arushi/090526_EvoAge/kg_formation/data_collection/harmonizome/refine_harmonizome/new_refine_harmonizome
Output (OUT_DIR) : /storage/Arushi/090526_EvoAge/kg_formation/processed_data/harmonizome
Config ready.


## 1. Load Lookup Dictionaries

In [7]:
# ── NCBI Gene Symbol → Description ───────────────────────
ENS_2NCBI = pd.read_csv(f'{DB_DIR}/ENSEMBL/ENSEMBLE_ID_2_NCBI_Gene_SYM.csv')
ENS_2NCBI = ENS_2NCBI[~ENS_2NCBI['name'].isna()]
NCBI_2ENS_dict = dict(zip(ENS_2NCBI['name'], ENS_2NCBI['initial_alias']))
del ENS_2NCBI

NCBI_ALL_GENE = pd.read_csv(f'{DB_DIR}/ncbi/Homo_sapiens.gene_info', sep='\t')
NCBI_ALL_GENE['ENSEMBLE_ID'] = NCBI_ALL_GENE['Symbol'].map(NCBI_2ENS_dict)
NCBI_Synbol_GENEname_dict   = dict(zip(NCBI_ALL_GENE['Symbol'],  NCBI_ALL_GENE['description']))
NCBI_ALL_GENEname_dict      = dict(zip(NCBI_ALL_GENE['GeneID'],  NCBI_ALL_GENE['description']))
NCBI_ALL_GENEIDname_dict    = dict(zip(NCBI_ALL_GENE['GeneID'],  NCBI_ALL_GENE['Symbol']))
del NCBI_ALL_GENE
print(f'NCBI dict: {len(NCBI_Synbol_GENEname_dict):,} genes')

NCBI dict: 193,352 genes


In [8]:
# ── BTO Tissue Ontology ───────────────────────────────────
BTO = pd.read_csv(f'{DB_DIR}/bto/BTO_ALL_COMBINED.csv')
BTO_dict     = dict(zip(BTO['name'], BTO['ID']))   # name → BTO ID
BTO_dict_rev = dict(zip(BTO['ID'],   BTO['name'])) # BTO ID → name
del BTO
print(f'BTO dict: {len(BTO_dict):,} entries')

BTO dict: 6,608 entries


In [9]:
# ── GO Ontology ───────────────────────────────────────────
GO = pd.read_csv(f'{DB_DIR}/MESH/MESH_GO_ID_NAME.csv')
GO['namespace'] = GO['namespace'].replace({
    'biological_process': 'Biological Process',
    'molecular_function': 'Molecular Function',
    'cellular_component': 'Cellular Component'
})
GO_dict           = dict(zip(GO['id'],   GO['name']))
GO_namespace_dict = dict(zip(GO['id'],   GO['namespace']))
del GO
print(f'GO dict: {len(GO_dict):,} terms')

GO dict: 47,995 terms


In [13]:
# ── HPO Phenotype Ontology ────────────────────────────────
import re
phenotype = pd.read_csv(f'{DB_DIR}/hpo/HPO.csv', sep=',')  # adjust path/format as needed
phenotype
HPOphenotype_name_dict = dict(zip(phenotype['id'], phenotype['name']))

def extract_umls(xref):
    if isinstance(xref, str):
        match = re.search(r'UMLS:[A-Za-z0-9]+', xref)
        return match.group(0).replace('UMLS:', '') if match else None
    return None

phenotype['UMLS_ID'] = phenotype['xref'].apply(extract_umls)
phenotype_HPO_UMLS   = phenotype[~phenotype['UMLS_ID'].isna()]
UMLS_2_HPO_phenotype_dict = dict(zip(phenotype_HPO_UMLS['UMLS_ID'], phenotype_HPO_UMLS['id']))

MedGen = pd.read_csv(f'{DB_DIR}/MESH/MeSH/MedGen_HPO_Mapping.txt', sep='|')
MedGen_CUID_Source_ID_dict  = dict(zip(MedGen['#CUI'],    MedGen['SDUI']))
MedGen_CUID_HPO_Source_ID_dict = dict(zip(MedGen['HpoStr'], MedGen['SDUI']))

phenotype_dict = dict(zip(phenotype['name'], phenotype['id']))
del phenotype
print(f'HPO dict: {len(HPOphenotype_name_dict):,} terms')

HPO dict: 19,484 terms


In [14]:
# ── DOID Disease Ontology ─────────────────────────────────
DO = pd.read_csv(f'{DB_DIR}/DO/All_DO_ref_files.csv')
DOID_disname_dict      = dict(zip(DO['id'],    DO['label']))
DOID_disAltID_dict     = dict(zip(DO['id'],    DO['xrefs']))
DOID_disname_DOID_dict = dict(zip(DO['label'], DO['id']))
del DO
print(f'DOID dict: {len(DOID_disname_dict):,} diseases')

DOID dict: 11,804 diseases


In [17]:
# ── UniProt ───────────────────────────────────────────────
Uniprot_names = pd.read_csv(f'{DB_DIR}/uniprot/UNIPROT_NCBI_ALL_MAPPED.csv')
Uni_NCBI      = Uniprot_names[~Uniprot_names['Gene_Name'].isna()]
Uni_NCBI_dict = dict(zip(Uni_NCBI['Gene_Name'], Uni_NCBI['Uniprot_ID']))

Uniprot_Recname     = pd.read_csv(f'{DB_DIR}/uniprot/Uniprot_ID_Recname.csv')
Uniprot_Recname_dict = dict(zip(Uniprot_Recname['AC'], Uniprot_Recname['RecName']))
del Uniprot_names, Uni_NCBI, Uniprot_Recname
print(f'UniProt dict: {len(Uni_NCBI_dict):,} gene→UniProt mappings')

/tmp/ipykernel_1764144/1987518986.py:2: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  Uniprot_names = pd.read_csv(f'{DB_DIR}/uniprot/UNIPROT_NCBI_ALL_MAPPED.csv')


UniProt dict: 26,176 gene→UniProt mappings


In [18]:
# ── Reactome Pathways ─────────────────────────────────────
reactome = pd.read_csv(f'{DB_DIR}/reactome/ReactomePathways.txt', sep='\t', header=None)
reactome = reactome[reactome[0].str.startswith('R-HSA')]
pathway_reactome_dict_rev = dict(zip(reactome[1], reactome[0]))  # name → R-HSA ID
del reactome
print(f'Reactome dict: {len(pathway_reactome_dict_rev):,} pathways')

Reactome dict: 2,734 pathways


In [20]:
# ── PubChem ───────────────────────────────────────────────
Pubchem = pd.read_pickle(f'{DB_DIR}/pubchem/combined_df.pkl')
Pubchem_CID_Smile_Dict  = dict(zip(Pubchem['PUBCHEM_COMPOUND_CID'], Pubchem['PUBCHEM_SMILES']))
Pubchem_IUPAC_CID_Dict  = dict(zip(Pubchem['PUBCHEM_COMPOUND_CID'], Pubchem['PUBCHEM_IUPAC_NAME']))
del Pubchem

pubchem2 = pd.read_csv(f'{DB_DIR}/pubchem/Pubchem_ID_DrugBank_Chebi.csv')
pubchem_DB   = pubchem2[~pubchem2['DRUGBANK_ID'].isna()]
DB2PC_dict   = dict(zip(pubchem_DB['DRUGBANK_ID'], pubchem_DB['ID']))  # DrugBankID → PubChemCID

Pubchem_Syn  = pd.read_csv(f'{DB_DIR}/pubchem/CID-Synonym-filtered', sep='\t', header=None)
Pubchem_Syn_fil_dict = dict(zip(Pubchem_Syn[1], Pubchem_Syn[0]))  # synonym → CID

Mesh_supp = pd.read_csv(f'{DB_DIR}/MESH/Mesh_Supplementary_Info.csv')
Mesh_pubchem_supp = Mesh_supp[~Mesh_supp['Pubchem_ID'].isna()].copy()
Mesh_pubchem_supp['Pubchem_ID'] = Mesh_pubchem_supp['Pubchem_ID'].astype(str).str.replace(r'\.0$','',regex=True)
MESH_pubchem_Supp_dict = dict(zip(Mesh_pubchem_supp['ID'], Mesh_pubchem_supp['Pubchem_ID']))
del pubchem2, pubchem_DB, Pubchem_Syn, Mesh_supp, Mesh_pubchem_supp
print('PubChem dicts ready')

/tmp/ipykernel_1764144/3208890631.py:7: DtypeWarning: Columns (1,2) have mixed types. Specify dtype option on import or set low_memory=False.
  pubchem2 = pd.read_csv(f'{DB_DIR}/pubchem/Pubchem_ID_DrugBank_Chebi.csv')


PubChem dicts ready


## 2. Gene–Phenotype

In [21]:
RAW_DIR

'/storage/Arushi/090526_EvoAge/kg_formation/data_collection/harmonizome/refine_harmonizome/new_refine_harmonizome'

In [54]:
# ── Files: GWASdb (HPOID), HPO, ClinVar, GWAS Catalog, HuGE ──
pheno_dfs = []

# GWASdb — has HPOID directly
df = pd.read_csv(f'{RAW_DIR}/gene-phenotype associations_GWASdb.csv')
df = df[df['weight'] == 1.0].rename(columns={'GeneSym':'Head','HPOID':'Tail'})
df['Head_detail_name'] = df['Head'].map(NCBI_Synbol_GENEname_dict)
df['Tail_detail_name'] = df['Tail'].map(HPOphenotype_name_dict)
df = df[~df['Tail_detail_name'].isna()]
pheno_dfs.append(df)
# pheno_dfs
print(df.shape)

# HPO/MPO — has HPOID directly
df = pd.read_csv(f'{RAW_DIR}/gene-disease associations_HPO.csv')  # We used This file because it had HP ID thatindicates phenotype
df = df[df['weight'] == 1.0].rename(columns={'GeneSym':'Head','DOID':'Tail'})
df['Head_detail_name'] = df['Head'].map(NCBI_Synbol_GENEname_dict)
df['Tail_detail_name'] = df['Tail'].map(HPOphenotype_name_dict)
df = df[~df['Tail_detail_name'].isna()]
pheno_dfs.append(df)
df
print(df.shape)

# ClinVar — phenotype name, map to HPO ID
df = pd.read_csv(f'{RAW_DIR}/gene-phenotype associations_ClinVar.csv')
df = df[df['weight'] == 1.0].rename(columns={'GeneSym':'Head','Phenotype':'Tail_detail_name'})
df['Head_detail_name'] = df['Head'].map(NCBI_Synbol_GENEname_dict)
df['Tail'] = df['Tail_detail_name'].map(phenotype_dict).fillna(
             df['Tail_detail_name'].map(MedGen_CUID_HPO_Source_ID_dict))
df = df[~df['Tail'].isna()]
pheno_dfs.append(df)
df
print(df.shape)


# GWAS Catalog — phenotype name, map to HPO ID
df = pd.read_csv(f'{RAW_DIR}/gene-phenotype associations_GWAS Catalog.csv')
df = df[df['weight'] == 1.0].rename(columns={'GeneSym':'Head','Phenotype':'Tail_detail_name'})
df['Head_detail_name'] = df['Head'].map(NCBI_Synbol_GENEname_dict)
df['Tail'] = df['Tail_detail_name'].map(phenotype_dict).fillna(
             df['Tail_detail_name'].map(MedGen_CUID_HPO_Source_ID_dict))
df = df[~df['Tail'].isna()]
pheno_dfs.append(df)
print(df.shape)


# HuGE Navigator — UMLS CUI → HPO
df = pd.read_csv(f'{RAW_DIR}/gene-phenotype associations_HuGE Navigator.csv')
df = df[df['weight'] == 1.0].rename(columns={'GeneSym':'Head','UMLS CUI':'UMLS_CUI'})
df['Tail'] = df['UMLS_CUI'].map(UMLS_2_HPO_phenotype_dict)
df = df[~df['Tail'].isna()]
df['Head_detail_name'] = df['Head'].map(NCBI_Synbol_GENEname_dict)
df['Tail_detail_name'] = df['Tail'].map(HPOphenotype_name_dict)
pheno_dfs.append(df)
print(df.shape)


# ── Merge & save ──
Gene_Phenotype = pd.concat(pheno_dfs, ignore_index=True)
Gene_Phenotype = finalize(Gene_Phenotype, 'Gene', 'Phenotype', 'NCBI_ID', 'HPO_ID')
save(Gene_Phenotype, 'Gene_Phenotype_Harmonizome.csv')
Gene_Phenotype.head(3)

(274574, 7)
(304975, 7)
(118, 6)
(898, 6)
(66901, 8)
  [SAVED] Gene_Phenotype_Harmonizome.csv  (617,022 rows)


,Head,Relation,Tail,Head_type,Tail_type,Source,KG_Source,Head_detail_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS
0,IFI16,Gene_Phenotype,HP:0010987,Gene,Phenotype,GWASdb,Harmonizome,interferon gamma inducible protein 16,Abnormal cellular immune system morphology,NCBI_ID,HPO_ID
1,CD84,Gene_Phenotype,HP:0010987,Gene,Phenotype,GWASdb,Harmonizome,CD84 molecule,Abnormal cellular immune system morphology,NCBI_ID,HPO_ID
2,FCRLA,Gene_Phenotype,HP:0010987,Gene,Phenotype,GWASdb,Harmonizome,Fc receptor like A,Abnormal cellular immune system morphology,NCBI_ID,HPO_ID


## 3. Gene–Gene (PPI + TF Targets + Co-expression)

In [55]:
PPI_FILES = [
    'protein-protein interaction_BioGRID.csv',
    'protein -protein associations_DIP.csv',
    'gene-gene assosiation_BIND.csv',
    'gene-gene assosiation_HumanCyc.csv',
    'gene-gene assosiation_PANTHER.csv',
    'gene-gene assosiation_PID.csv',
    'gene-gene assosiation_Reactome.csv',
    'gene-gene assosiation_HPRD.csv',
    'gene-gene assosiation_IntAct.csv',
    'gene-gene assosiation_KEGG.csv',
    'gene-gene assosiation_HubProteins.csv',
    'gene-gene assosiation_NURSA.csv',
    'gene-gene assosiation_KEA.csv',
    'gene-gene assosiation_PhosphoSitePlus_Kinase.csv',
    'gene-gene associations_TRANSFAC .csv',
    'gene-gene associations_TRANSFAC .csv_1.csv',
    'gene-co-expressed gene associations_MSigDB.csv',
    'gene-gene associations_ENCODE_TF.csv',
    'gene-gene associations_JASPAR.csv',
    'gene-gene associations_TRANSFAC_curated.csv',
    'gene-gene associations_TRANSFAC_predicted.csv',
]

gg_dfs = []
for fname in PPI_FILES:
    path = os.path.join(RAW_DIR, fname)
    if not os.path.exists(path): continue
    df = pd.read_csv(path)
    df = df[df['weight'] == 1.0]
    df = df.rename(columns={'GeneSym':'Head', 'GeneSym_1':'Tail'})
    df['Head_detail_name'] = df['Head'].map(NCBI_Synbol_GENEname_dict)
    df['Tail_detail_name'] = df['Tail'].map(NCBI_Synbol_GENEname_dict)
    df = df[~df['Tail_detail_name'].isna()]
    # Remove self-loops
    df = df[df['Head'] != df['Tail']]
    print(df.shape)
    gg_dfs.append(df)

Gene_Gene = pd.concat(gg_dfs, ignore_index=True)
Gene_Gene = finalize(Gene_Gene, 'Gene', 'Gene', 'NCBI_ID', 'NCBI_ID')
save(Gene_Gene, 'Gene_Gene_Harmonizome.csv')
print(f'Unique gene pairs: {len(Gene_Gene):,}')
Gene_Gene.head(3)

(303732, 7)
(14118, 7)
(28274, 7)
(782, 7)
(42244, 7)
(25751, 7)
(211052, 7)
(115474, 7)
(148464, 7)
(10, 7)
(56748, 7)
(3982, 7)
(11842, 7)
(5787, 7)
(386678, 7)
(420516, 7)
(40464, 7)
(1629712, 7)
(148047, 7)
(100560, 7)
(229490, 7)
  [SAVED] Gene_Gene_Harmonizome.csv  (3,603,960 rows)
Unique gene pairs: 3,603,960


,Head,Relation,Tail,Head_type,Tail_type,Source,KG_Source,Head_detail_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS
0,WWOX,Gene_Gene,MYH16,Gene,Gene,BioGRID Protein-Protein Interactions,Harmonizome,WW domain containing oxidoreductase,myosin heavy chain 16,NCBI_ID,NCBI_ID
1,WWOX,Gene_Gene,RPS15AP21,Gene,Gene,BioGRID Protein-Protein Interactions,Harmonizome,WW domain containing oxidoreductase,ribosomal protein S15a pseudogene 21,NCBI_ID,NCBI_ID
2,WWOX,Gene_Gene,A2ML1,Gene,Gene,BioGRID Protein-Protein Interactions,Harmonizome,WW domain containing oxidoreductase,alpha-2-macroglobulin like 1,NCBI_ID,NCBI_ID


## 4. Gene–Tissue

In [97]:
BTO_dict.get('pituitary')

In [98]:
# Files with Tissue column (name → BTO ID via BTO_dict)
TISSUE_NAME_FILES = [
    ('gene_tissue_biogrkan.csv',                            'Tissue'),
    ('gene-tisue_Allen Brain Atlas.csv',                    'Structure'),
    ('gene-tissue_Allen Brain Atlas.csv',                   'StructureName'),
    ('gene-tissue sample associations_GTEx.csv',            'Tissue'),
    ('gene-tissue sample associations_Allen Brain Atlas.csv_1.csv', 'Structure_Age_Gender_DonorID'),
    ('gene-tissue sample associations_TCGA .csv',           'Cancer Name_Cancer Acronym'),
    ('gene-tissue associations_Allen Brain Atlas.csv',      'Structure'),
    ('gene-tissue associations_HPA.csv',                    'Tissue'),
    ('gene-tissue associations_HPA.csv_1.csv',              'Tissue'),
    ('gene-tissue associations_BioGPS_Human.csv',           'Tissue'),
    ('gene-tissue associations_BioGPS_Mouse.csv',           'Tissue'),
]

# Files with BTO ID directly (BTO column, use BTO_dict_rev for name)
TISSUE_BTO_FILES = [
    ('gene-tissue associations_TISSUES .csv',       'BTO', 'Tissue'),
    ('gene-tissue associations_TISSUES .csv_1.csv', 'BTO', 'Tissue'),
    ('gene-tissue associations_TISSUES .csv_2.csv', 'BTO', 'Tissue'),
]

tissue_dfs = []

# Process tissue-name files (map name → BTO ID)
for fname, tissue_col in TISSUE_NAME_FILES:
    path = os.path.join(RAW_DIR, fname)
    if not os.path.exists(path): continue
    df = pd.read_csv(path)
    df = df[df['weight'] == 1.0].rename(columns={'GeneSym':'Head'})
    df['Head_detail_name'] = df['Head'].map(NCBI_Synbol_GENEname_dict)
    df['Tail_detail_name'] = df[tissue_col]          # human-readable name
    df['Tail'] = df[tissue_col].map(BTO_dict)         # BTO ID
    df = df[~df['Tail'].isna()]
    print(fname)
    print(df.shape)
    tissue_dfs.append(df)

# Process BTO-ID files (already have BTO ID, get name from BTO_dict_rev)
for fname, bto_col, name_col in TISSUE_BTO_FILES:
    path = os.path.join(RAW_DIR, fname)
    if not os.path.exists(path): continue
    df = pd.read_csv(path)
    df = df[df['weight'] == 1.0].rename(columns={'GeneSym':'Head'})
    df['Head_detail_name'] = df['Head'].map(NCBI_Synbol_GENEname_dict)
    df['Tail'] = df[bto_col]                           # BTO ID already
    df['Tail_detail_name'] = df[name_col]              # tissue name
    df = df[~df['Tail'].isna()]
    print(df.shape)
    tissue_dfs.append(df)

Gene_Tissue = pd.concat(tissue_dfs, ignore_index=True)
Gene_Tissue = finalize(Gene_Tissue, 'Gene', 'Tissue', 'NCBI_ID', 'BTO_ID')
save(Gene_Tissue, 'Gene_Tissue_Harmonizome.csv')
Gene_Tissue.head(3)

gene_tissue_biogrkan.csv
(43234, 7)
gene-tisue_Allen Brain Atlas.csv
(5425, 7)
gene-tissue_Allen Brain Atlas.csv
(20647, 7)
gene-tissue sample associations_GTEx.csv
(3818234, 7)
gene-tissue sample associations_Allen Brain Atlas.csv_1.csv
(0, 7)
gene-tissue sample associations_TCGA .csv
(0, 7)
gene-tissue associations_Allen Brain Atlas.csv
(5425, 7)
gene-tissue associations_HPA.csv
(33478, 7)
gene-tissue associations_HPA.csv_1.csv
(79459, 7)
gene-tissue associations_BioGPS_Human.csv
(1552, 7)
gene-tissue associations_BioGPS_Mouse.csv
(27057, 7)
(357465, 8)
(274154, 8)
(1836577, 8)
  [SAVED] Gene_Tissue_Harmonizome.csv  (2,711,071 rows)


,Head,Relation,Tail,Head_type,Tail_type,Source,KG_Source,Head_detail_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS
0,LINC00208,Gene_Tissue,BTO:0000562,Gene,Tissue,GTEx,Harmonizome,long intergenic non-protein coding RNA 208,heart,NCBI_ID,BTO_ID
2,LIN9,Gene_Tissue,BTO:0000562,Gene,Tissue,GTEx,Harmonizome,lin-9 DREAM MuvB core complex component,heart,NCBI_ID,BTO_ID
3,PPFIBP1,Gene_Tissue,BTO:0000562,Gene,Tissue,GTEx,Harmonizome,PPFIA binding protein 1,heart,NCBI_ID,BTO_ID


## 5. Gene–Cellular Component (GO CC)

In [57]:
CC_FILES = [
    'gene-cellular component associations_COMPARTMENTS.csv',
    'gene-cellular component associations_GO.csv',
    'gene-cellular component associations_LOCATE.csv',
    'gene-cellular component associations_LOCATE.csv_1.csv',
]

cc_dfs = []
for fname in CC_FILES:
    path = os.path.join(RAW_DIR, fname)
    if not os.path.exists(path): continue
    df = pd.read_csv(path)
    df = df[df['weight'] == 1.0].rename(columns={'GeneSym':'Head', 'GO':'Tail'})
    df['Head_detail_name'] = df['Head'].map(NCBI_Synbol_GENEname_dict)
    df['Tail_detail_name'] = df['Tail'].map(GO_dict)
    df['Tail_type_specific'] = df['Tail'].map(GO_namespace_dict)
    df = df[~df['Tail_detail_name'].isna()]
    print(df.shape)
    cc_dfs.append(df)

Gene_CC = pd.concat(cc_dfs, ignore_index=True)
Gene_CC = finalize(Gene_CC, 'Gene', 'Cellular Component', 'NCBI_ID', 'Quick_GO')
save(Gene_CC, 'Gene_CellularComponent_Harmonizome.csv')
Gene_CC.head(3)

(311508, 9)
(242137, 9)
(78596, 9)
(154700, 9)
  [SAVED] Gene_CellularComponent_Harmonizome.csv  (743,987 rows)


,Head,Relation,Tail,Head_type,Tail_type,Source,KG_Source,Head_detail_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS
0,PI4K2A,Gene_Cellular Component,GO:0044218,Gene,Cellular Component,COMPARTMENTS Curated Protein Localization Evid...,Harmonizome,phosphatidylinositol 4-kinase type 2 alpha,other organism cell membrane,NCBI_ID,Quick_GO
1,PI4K2A,Gene_Cellular Component,GO:0044221,Gene,Cellular Component,COMPARTMENTS Curated Protein Localization Evid...,Harmonizome,phosphatidylinositol 4-kinase type 2 alpha,host cell synapse,NCBI_ID,Quick_GO
2,PI4K2A,Gene_Cellular Component,GO:0033644,Gene,Cellular Component,COMPARTMENTS Curated Protein Localization Evid...,Harmonizome,phosphatidylinositol 4-kinase type 2 alpha,host cell membrane,NCBI_ID,Quick_GO


## 6. Gene–Disease

In [63]:
# Files where target=Disease name, DOID column has the ID
DISEASE_DOID_FILES = [
    'gene-disease associations_DISEASES.csv',
    'gene-disease associations_DISEASES.csv_1.csv',
    'gene-disease associations_DISEASES.csv_2.csv',
    'gene-disease associations_GWASdb.csv',
    #'gene-disease associations_HPO.csv',
    'gene-disease associations_dbGAP.csv',
]

# Files where Disease is a name — map via DOID_disname_DOID_dict
DISEASE_NAME_FILES = [
    'gene-disease associations_GAD.csv',
    'gene-disease associations_GAD_high.csv',
    'gene-disease associations_PhosphoSitePlus.csv',
    'gene-disease associations_CTD.csv',
    'gene-disease associations_OMIM.csv',
    'gene-disease perturbation associations_GEO .csv',
]

dis_dfs = []

for fname in DISEASE_DOID_FILES:
    path = os.path.join(RAW_DIR, fname)
    if not os.path.exists(path): continue
    df = pd.read_csv(path)
    df = df[df['weight'] == 1.0].rename(columns={'GeneSym':'Head', 'DOID':'Tail', 'Disease':'Tail_detail_name'})
    df['Head_detail_name'] = df['Head'].map(NCBI_Synbol_GENEname_dict)
    df = df[~df['Tail'].isna() & ~df['Tail_detail_name'].isna()]
    dis_dfs.append(df)

for fname in DISEASE_NAME_FILES:
    path = os.path.join(RAW_DIR, fname)
    if not os.path.exists(path): continue
    df = pd.read_csv(path)
    df = df[df['weight'] == 1.0].rename(columns={'GeneSym':'Head', 'Disease':'Tail_detail_name'})
    df['Head_detail_name'] = df['Head'].map(NCBI_Synbol_GENEname_dict)
    df['Tail'] = df['Tail_detail_name'].map(DOID_disname_DOID_dict)
    df = df[~df['Tail'].isna()]
    print(df.shape)
    dis_dfs.append(df)

Gene_Disease = pd.concat(dis_dfs, ignore_index=True)
Gene_Disease = finalize(Gene_Disease, 'Gene', 'Disease', 'NCBI_ID', 'DOID')
save(Gene_Disease, 'Gene_Disease_Harmonizome.csv')
Gene_Disease.head(3)

(13398, 7)
(2534, 7)
(204, 7)
(102, 7)
(635, 7)
(600, 6)
  [SAVED] Gene_Disease_Harmonizome.csv  (1,083,855 rows)


,Head,Relation,Tail,Head_type,Tail_type,Source,KG_Source,Head_detail_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS
0,VHL,Gene_Disease,DOID:5241,Gene,Disease,DISEASES Curated Gene-Disease Assocation Evide...,Harmonizome,von Hippel-Lindau tumor suppressor,hemangioblastoma,NCBI_ID,DOID
1,VHL,Gene_Disease,DOID:14175,Gene,Disease,DISEASES Curated Gene-Disease Assocation Evide...,Harmonizome,von Hippel-Lindau tumor suppressor,von hippel-lindau disease,NCBI_ID,DOID
2,VHL,Gene_Disease,DOID:255,Gene,Disease,DISEASES Curated Gene-Disease Assocation Evide...,Harmonizome,von Hippel-Lindau tumor suppressor,hemangioma,NCBI_ID,DOID


## 7. Gene–ChemicalEntity (Drug + Chemical)

In [61]:
chem_dfs = []

# CTD Gene-Chemical — target=Chemical name, target_desc=PubchemCID (MESH style)
df = pd.read_csv(f'{RAW_DIR}/gene-chemical associations_CTD.csv')
df = df[df['weight'] == 1.0].rename(columns={'GeneSym':'Head', 'PubchemCID':'Tail_ID', 'Chemical':'Tail_detail_name'})
df['Head_detail_name'] = df['Head'].map(NCBI_Synbol_GENEname_dict)
df['Tail'] = df['Tail_ID'].map(MESH_pubchem_Supp_dict)
df = df[~df['Tail'].isna()]
df['Tail_IUPAC']  = df['Tail'].map(Pubchem_IUPAC_CID_Dict)
df['Tail_Smiles'] = df['Tail'].map(Pubchem_CID_Smile_Dict)
df = df[~df['Tail_IUPAC'].isna()]
df['Tail_ID_IS'] = 'Pubchem'
print(df.shape)
chem_dfs.append(df)

# DrugBank Drug Targets — DrugBankID → PubChem CID
df = pd.read_csv(f'{RAW_DIR}/gene-drug associations_DrugBank.csv')
df = df[df['weight'] == 1.0].rename(columns={'GeneSym':'Head', 'DrugBankID':'DB_ID', 'DrugName':'Tail_detail_name'})
df['Head_detail_name'] = df['Head'].map(NCBI_Synbol_GENEname_dict)
df['Tail'] = df['DB_ID'].map(DB2PC_dict).fillna(df['DB_ID'])
df['Tail'] = df['Tail'].astype(str).str.replace(r'\.0$', '', regex=True)
df['Tail_IUPAC']  = df['Tail'].map(Pubchem_IUPAC_CID_Dict)
df['Tail_Smiles'] = df['Tail'].map(Pubchem_CID_Smile_Dict)
df['Tail_ID_IS']  = np.where(df['Tail'].str.startswith('DB'), 'Drugbank', 'Pubchem')
print(df.shape)
chem_dfs.append(df)

# Guide to Pharmacology Chemical Ligands
df = pd.read_csv(f'{RAW_DIR}/gene-drug associations_GtP.csv')
df = df[df['weight'] == 1.0].rename(columns={'GeneSym':'Head', 'DrugName':'Tail_detail_name', 'DrugBankID':'Tail'})
df['Head_detail_name'] = df['Head'].map(NCBI_Synbol_GENEname_dict)
df['Tail_ID_IS'] = 'Pubchem'
print(df.shape)
chem_dfs.append(df)

Gene_Chemical = pd.concat(chem_dfs, ignore_index=True)
Gene_Chemical = finalize(Gene_Chemical, 'Gene', 'ChemicalEntity', 'NCBI_ID', 'Pubchem')
save(Gene_Chemical, 'Gene_ChemicalEntity_Harmonizome.csv')
Gene_Chemical.head(3)

(11801, 10)
(15261, 10)
(9380, 7)
  [SAVED] Gene_ChemicalEntity_Harmonizome.csv  (36,203 rows)


,Head,Relation,Tail,Head_type,Tail_type,Source,KG_Source,Head_detail_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS
0,TBXA2R,Gene_ChemicalEntity,6438346,Gene,ChemicalEntity,CTD,Harmonizome,thromboxane A2 receptor,7-(3-(3-hydroxy-4-(4'-iodophenoxy)-1-butenyl)-...,NCBI_ID,Pubchem
1,TBXA2R,Gene_ChemicalEntity,5311448,Gene,ChemicalEntity,CTD,Harmonizome,thromboxane A2 receptor,SQ 29548,NCBI_ID,Pubchem
2,EDN3,Gene_ChemicalEntity,44298976,Gene,ChemicalEntity,CTD,Harmonizome,endothelin 3,S 145,NCBI_ID,Pubchem


In [65]:
# Gene_Chemical[Gene_Chemical['Tail'].str.startswith('DB')]

## 8. Gene–Protein (UniProt)

In [62]:
PROTEIN_FILES = [
    'gene-interacting protein associations_NURSA.csv',
    'gene-interacting protein associations_Pathway Commons.csv',
    'gene-interacting protein associations_ReconX.csv',
    'gene-ligand (protein) associations_Guide to Pharmacology.csv',
]

prot_dfs = []
for fname in PROTEIN_FILES:
    path = os.path.join(RAW_DIR, fname)
    if not os.path.exists(path): continue
    df = pd.read_csv(path)
    df = df[df['weight'] == 1.0].rename(columns={'GeneSym':'Head', 'GeneSym_1':'Gene_partner'})
    df['Head_detail_name'] = df['Head'].map(NCBI_Synbol_GENEname_dict)
    # Map gene symbol → UniProt ID
    df['Tail'] = df['Gene_partner'].map(Uni_NCBI_dict)
    df = df[~df['Tail'].isna()]
    df['Tail_detail_name'] = df['Tail'].map(Uniprot_Recname_dict)
    df = df[~df['Tail_detail_name'].isna()]
    print(df.shape)
    prot_dfs.append(df)

Gene_Protein = pd.concat(prot_dfs, ignore_index=True)
Gene_Protein = finalize(Gene_Protein, 'Gene', 'Protein', 'NCBI_ID', 'Uniprot_ID')
save(Gene_Protein, 'Gene_Protein_Harmonizome.csv')
Gene_Protein.head(3)

(0, 7)
(300135, 7)
(398, 7)
(121, 7)
  [SAVED] Gene_Protein_Harmonizome.csv  (287,229 rows)


,Head,Relation,Tail,Head_type,Tail_type,Source,KG_Source,Head_detail_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS
0,HNF4A,Gene_Protein,Q9UGM6,Gene,Protein,Pathway Commons Protein-Protein Interactions,Harmonizome,hepatocyte nuclear factor 4 alpha,"Tryptophan--tRNA ligase, mitochondrial",NCBI_ID,Uniprot_ID
1,UBC,Gene_Protein,Q9UGM6,Gene,Protein,Pathway Commons Protein-Protein Interactions,Harmonizome,ubiquitin C,"Tryptophan--tRNA ligase, mitochondrial",NCBI_ID,Uniprot_ID
2,NRF1,Gene_Protein,Q9UGM6,Gene,Protein,Pathway Commons Protein-Protein Interactions,Harmonizome,nuclear respiratory factor 1,"Tryptophan--tRNA ligase, mitochondrial",NCBI_ID,Uniprot_ID


## 9. Gene–Metabolite (PubChem)

In [83]:
# HMDB: HMDB ACC.1 → PubChem CID via Metabolite_Iupac_dict
Metabolite = pd.read_csv(f'{DB_DIR}/hmdb/HMDBoutput_New.csv')
Metabolite['Pubchem_final'] = Metabolite['Pubchem_final'].astype(str).str.replace(r'\.0$','',regex=True)
Metabolite_Iupac = Metabolite[Metabolite['Pubchem_final'] != 'nan']
Metabolite_Iupac_dict = dict(zip(Metabolite_Iupac['accession'], Metabolite_Iupac['Pubchem_final']))

def format_hmdb(v):
    if isinstance(v, str) and v.startswith('HMDB'):
        return f"HMDB{int(v[4:]):07d}"
    return v

df = pd.read_csv(f'{RAW_DIR}/gene-metabolite associations_HMDB.csv')
df = df[df['weight'] == 1.0].rename(columns={'GeneSym':'Head', 'Metabolite':'Tail_common_name'})
df['HMDB_ACC'] = df['HMDB ACC.1'].apply(format_hmdb)
df['Head_detail_name'] = df['Head'].map(NCBI_Synbol_GENEname_dict)
df['Tail'] = df['HMDB_ACC'].map(Metabolite_Iupac_dict)   # → PubChem CID
df['Tail_detail_name'] = df['Tail'].map(Pubchem_IUPAC_CID_Dict)
df['Tail_Smiles']      = df['Tail'].map(Pubchem_CID_Smile_Dict)
df = df[~df['Tail_detail_name'].isna()]
print(df.shape)

Gene_Metabolite = finalize(df, 'Gene', 'ChemicalEntity', 'NCBI_ID', 'Pubchem')
save(Gene_Metabolite, 'Gene_Metabolite_Harmonizome.csv')
del Metabolite, Metabolite_Iupac
Gene_Metabolite.head(3)

(781644, 10)
  [SAVED] Gene_Metabolite_Harmonizome.csv  (770,740 rows)


,Head,Relation,Tail,Head_type,Tail_type,Source,KG_Source,Head_detail_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS
0,EPHX1,Gene_ChemicalEntity,853019,Gene,ChemicalEntity,HMDB,Harmonizome,epoxide hydrolase 1,"(1R,2R)-1,2-diphenylethane-1,2-diol",NCBI_ID,Pubchem
1,ACAA1,Gene_ChemicalEntity,71448920,Gene,ChemicalEntity,HMDB,Harmonizome,acetyl-CoA acyltransferase 1,"S-[2-[3-[[(2R)-4-[[[(2R,3S,4R,5R)-5-(6-aminopu...",NCBI_ID,Pubchem
2,CYP2C19,Gene_ChemicalEntity,439250,Gene,ChemicalEntity,HMDB,Harmonizome,cytochrome P450 family 2 subfamily C member 19,(4S)-1-methyl-4-prop-1-en-2-ylcyclohexene,NCBI_ID,Pubchem


## 10. Gene–Molecular Function (GO MF)

In [84]:
mf_dfs = []
for fname in ['gene-molecular function associations_GO.csv',
              'gene-biological process associations_GO.csv']:
    path = os.path.join(RAW_DIR, fname)
    if not os.path.exists(path): continue
    df = pd.read_csv(path)
    df = df[df['weight'] == 1.0].rename(
        columns={'GeneSym':'Head', 'relation_source':'Tail'})
    df['Head_detail_name'] = df['Head'].map(NCBI_Synbol_GENEname_dict)
    df['Tail_detail_name'] = df['Tail'].map(GO_dict)
    df['Tail_type_specific'] = df['Tail'].map(GO_namespace_dict)
    df = df[~df['Tail_detail_name'].isna()]
    mf_dfs.append(df)

Gene_MF = pd.concat(mf_dfs, ignore_index=True)
Gene_MF = finalize(Gene_MF, 'Gene', 'MolecularFunction', 'NCBI_ID', 'Quick_GO')
save(Gene_MF, 'Gene_MolecularFunction_Harmonizome.csv')
Gene_MF.head(3)

  [SAVED] Gene_MolecularFunction_Harmonizome.csv  (1,078,295 rows)


,Head,Relation,Tail,Head_type,Tail_type,Source,KG_Source,Head_detail_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS
0,ABCA1,Gene_MolecularFunction,GO:0042626,Gene,MolecularFunction,GO MF,Harmonizome,ATP binding cassette subfamily A member 1,ATPase-coupled transmembrane transporter activity,NCBI_ID,Quick_GO
1,ABCA10,Gene_MolecularFunction,GO:0042626,Gene,MolecularFunction,GO MF,Harmonizome,ATP binding cassette subfamily A member 10,ATPase-coupled transmembrane transporter activity,NCBI_ID,Quick_GO
2,ABCA12,Gene_MolecularFunction,GO:0042626,Gene,MolecularFunction,GO MF,Harmonizome,ATP binding cassette subfamily A member 12,ATPase-coupled transmembrane transporter activity,NCBI_ID,Quick_GO


## 11. Gene–Pathway (Reactome)

In [85]:
df = pd.read_csv(f'{RAW_DIR}/gene-pathway associations_Reactome.csv')
df = df[df['weight'] == 1.0].rename(columns={'GeneSym':'Head', 'Pathway':'Tail_detail_name'})
df['Head_detail_name'] = df['Head'].map(NCBI_Synbol_GENEname_dict)
# Map pathway name → Reactome ID
df['Tail'] = df['Tail_detail_name'].map(pathway_reactome_dict_rev)
df = df[~df['Tail'].isna()]

Gene_Pathway = finalize(df, 'Gene', 'Pathway', 'NCBI_ID', 'Reactome')
save(Gene_Pathway, 'Gene_Pathway_Harmonizome.csv')
Gene_Pathway.head(3)

  [SAVED] Gene_Pathway_Harmonizome.csv  (62,384 rows)


,Head,Relation,Tail,Head_type,Tail_type,Source,KG_Source,Head_detail_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS
0,RAE1,Gene_Pathway,R-HSA-71387,Gene,Pathway,Reactome,Harmonizome,ribonucleic acid export 1,Metabolism of carbohydrates,NCBI_ID,Reactome
1,TPR,Gene_Pathway,R-HSA-71387,Gene,Pathway,Reactome,Harmonizome,"translocated promoter region, nuclear basket p...",Metabolism of carbohydrates,NCBI_ID,Reactome
2,AAAS,Gene_Pathway,R-HSA-71387,Gene,Pathway,Reactome,Harmonizome,aladin WD repeat nucleoporin,Metabolism of carbohydrates,NCBI_ID,Reactome


## 12. Summary

In [86]:
import os
from glob import glob

print('\n' + '='*55)
print('FINAL OUTPUT SUMMARY')
print('='*55)
total = 0
for f in sorted(glob(os.path.join(OUT_DIR, '*.csv'))):
    df = pd.read_csv(f)
    print(f'  {os.path.basename(f):<45} {len(df):>10,} rows')
    total += len(df)
print(f'  {"─"*55}')
print(f'  Total triples: {total:,}')


FINAL OUTPUT SUMMARY
  Gene_CellularComponent_Harmonizome.csv           743,987 rows
  Gene_ChemicalEntity_Harmonizome.csv               36,203 rows
  Gene_Disease_Harmonizome.csv                   1,083,855 rows
  Gene_Gene_Harmonizome.csv                      3,603,960 rows
  Gene_Metabolite_Harmonizome.csv                  770,740 rows
  Gene_MolecularFunction_Harmonizome.csv         1,078,295 rows
  Gene_Pathway_Harmonizome.csv                      62,384 rows
  Gene_Phenotype_Harmonizome.csv                   617,022 rows
  Gene_Protein_Harmonizome.csv                     287,229 rows
  Gene_Tissue_Harmonizome.csv                    2,711,071 rows
  ───────────────────────────────────────────────────────
  Total triples: 10,994,746


In [99]:
"""
deduplicate_harmonizome.py
===========================
Removes duplicate rows based on (Head, Relation, Tail) — keeps first occurrence.
Overwrites files in-place in OUT_PATH.

Usage:
    python deduplicate_harmonizome.py
"""

import os
import pandas as pd
from glob import glob

OUT_PATH = "/storage/Arushi/090526_EvoAge/kg_formation/processed_data/harmonizome/"

files = sorted(glob(os.path.join(OUT_PATH, "*.csv")))
print(f"\nFound {len(files)} files in {OUT_PATH}\n")
print(f"{'File':<50} {'Before':>10} {'After':>10} {'Removed':>10}")
print("─" * 85)

total_before = total_after = 0

for fpath in files:
    df = pd.read_csv(fpath)
    before = len(df)
    df = df.drop_duplicates(subset=["Head", "Relation", "Tail"], keep="first")
    after = len(df)
    df.to_csv(fpath, index=False)

    total_before += before
    total_after  += after
    print(f"{os.path.basename(fpath):<50} {before:>10,} {after:>10,} {before-after:>10,}")

print("─" * 85)
print(f"{'TOTAL':<50} {total_before:>10,} {total_after:>10,} {total_before-total_after:>10,}")
print("\nDone — files saved in place.")


Found 10 files in /storage/Arushi/090526_EvoAge/kg_formation/processed_data/harmonizome/

File                                                   Before      After    Removed
─────────────────────────────────────────────────────────────────────────────────────
Gene_CellularComponent_Harmonizome.csv                743,987    375,817    368,170
Gene_ChemicalEntity_Harmonizome.csv                    36,203     27,588      8,615
Gene_Disease_Harmonizome.csv                        1,083,855    974,298    109,557
Gene_Gene_Harmonizome.csv                           3,603,960  2,750,021    853,939
Gene_Metabolite_Harmonizome.csv                       770,740    770,740          0
Gene_MolecularFunction_Harmonizome.csv              1,078,295  1,078,295          0
Gene_Pathway_Harmonizome.csv                           62,384     62,384          0
Gene_Phenotype_Harmonizome.csv                        617,022    594,596     22,426
Gene_Protein_Harmonizome.csv                          287,229    28